# Identifying an equivalent-circuit battery model from a pulse test

Equivalent-circuit models describe a battery cell without any knowledge of its
internal chemistry: a voltage source for the open-circuit voltage $U_\mathrm{OCV}(s)$
as a function of the state of charge $s$, a series resistance $R_0$ for the
instantaneous (ohmic) response, and resistor-capacitor branches for the slower
polarisation. With one branch,

$$U=U_\mathrm{OCV}(s)-R_0\,I-u_1,\qquad
  \frac{du_1}{dt}=\frac{I}{C_1}-\frac{u_1}{R_1C_1},\qquad
  \frac{ds}{dt}=-\frac{I}{Q},$$

with the current $I$ positive on discharge, the capacity $Q$ and the relaxation time
$\tau_1=R_1C_1$. A pulse test (current pulses with rests in between) separates the
parameters: the voltage jumps at the pulse edges give $R_0$, the slow relaxation after
them $R_1$ and $\tau_1$.

OneSimulation solves the circuit (voltage source, $R_0$, the $R_1C_1$ branch, current
load) with its circuit solver. The `SimulationStudyService` runs it for trial values of
$(R_0,R_1,\ln\tau_1)$, measures the mean squared deviation (MSD) of the terminal voltage
from the data in mV², and its projected-gradient optimizer (finite-difference gradients
from real runs, backtracking line search, bounds) minimises it. The optimizer steps
along the plain gradient, so the parameters are chosen such that a unit change moves
the voltage by a few millivolts: resistances in mΩ and the logarithm of $\tau_1$.

**Reference.** G. L. Plett, *Battery Management Systems, Vol. I: Battery Modeling*,
Artech House 2015, ch. 2; the pulse test follows the hybrid pulse power
characterisation idea of the *FreedomCAR Battery Test Manual* (INL 2003). The data are
synthetic: generated by the same circuit from the "true" values $R_0=15$ mΩ,
$R_1=10$ mΩ, $\tau_1=30$ s with normally distributed noise of 1 mV. A correct
identification recovers these values within the uncertainty the noise allows and
leaves residuals of the size of the noise.

In [ ]:
import numpy as np

from onesim.physics.systems import Circuit, CircuitCapacitor, CircuitCurrentSource, CircuitResistor, CircuitVoltageSource
from onesim.simulation.studies import SimulationStudyService, StudyParameter
from onesim.visualization import theme, plots

theme.set()

## Cell and pulse test

A 3 Ah cell, starting at 80 % state of charge. The test applies a 10 s discharge pulse
at 2C (6 A), 40 s rest, a 10 s charge pulse at 1.5C, 40 s rest, then the same at a
lower state of charge after a 300 s discharge at 1C; the voltage is sampled every
second. The open-circuit voltage curve is taken as known (measured beforehand).

In [ ]:
capacity, start, noise = 3.0 * 3600.0, 0.80, 1e-3
open_circuit = lambda s: 3.45 + 0.55 * s + 0.08 * np.log(s / (1.0 - s)) / 4.0
steps = np.array([(0, 6.0), (10, 0.0), (50, -4.5), (60, 0.0), (100, 3.0), (400, 0.0), (500, 6.0), (510, 0.0), (550, -4.5), (560, 0.0)])
load = lambda t: steps[np.searchsorted(steps[:, 0], t, side="right") - 1, 1]
times = np.arange(0.0, 661.0, 1.0)
state = start - np.r_[0.0, np.cumsum([load(t) for t in times[:-1]])] / capacity
print(f"state of charge from {state[0]:.3f} to {state[-1]:.3f}, open-circuit voltage from {open_circuit(state[0]):.4f} V to {open_circuit(state[-1]):.4f} V")

## Circuit model

`pulse_test(values)` builds the circuit for one parameter set; the voltage source
follows the open-circuit voltage along the coulomb-counted state of charge.

In [ ]:
pulse_test = lambda values: Circuit(components=(
    CircuitVoltageSource(name="ocv", positive="p", negative="0", voltage=lambda t: open_circuit(np.interp(t, times, state))),
    CircuitResistor(name="r0", positive="p", negative="a", resistance=1e-3 * values["r0"]),
    CircuitResistor(name="r1", positive="a", negative="t", resistance=1e-3 * values["r1"]),
    CircuitCapacitor(name="c1", positive="a", negative="t", capacitance=np.exp(values["ln_tau1"]) / (1e-3 * values["r1"]), initial_voltage=0.0),
    CircuitCurrentSource(name="load", positive="t", negative="0", current=load)))
true = {"r0": 15.0, "r1": 10.0, "ln_tau1": np.log(30.0)}
rng = np.random.default_rng(2026)
data = pulse_test(true).solve(times=times, max_step=0.25).voltages["t"] + rng.normal(0.0, noise, times.size)

## Identification

Bounds: resistances between 1 and 50 mΩ, relaxation time between 2 and 200 s. The
optimizer starts at $R_0=30$ mΩ, $R_1=4$ mΩ, $\tau_1=80$ s, uses finite-difference steps
of $10^{-6}$ of each interval and stops when the projected gradient of the MSD falls
below $10^{-3}$ mV² per unit parameter or the line search finds no further decrease.

In [ ]:
study = SimulationStudyService(
    (StudyParameter("r0", 1.0, 50.0), StudyParameter("r1", 1.0, 50.0), StudyParameter("ln_tau1", np.log(2.0), np.log(200.0))),
    pulse_test,
    lambda circuit: {"msd": float(np.mean((1e3 * (circuit.voltages["t"] - data)) ** 2))},
    solve=lambda circuit: circuit.solve(times=times, max_step=0.25))
start_values = {"r0": 30.0, "r1": 4.0, "ln_tau1": np.log(80.0)}
best = study.optimize(("msd",), start=start_values, step=1e-6, tolerance=1e-3, maximum_iterations=150, maximum_runs=2500)
fitted = pulse_test(best.parameters).solve(times=times, max_step=0.25)

## Comparison with the true values

The uncertainty follows from the Gauss-Newton covariance $(J^TJ)^{-1}$ of the weighted
residuals, with the sensitivities $J$ from one more run per parameter.

In [ ]:
names = ("r0", "r1", "ln_tau1")
sensitivity = np.column_stack([(pulse_test(dict(best.parameters) | {name: best.parameters[name] + 1e-3}).solve(times=times, max_step=0.25).voltages["t"]
                                - fitted.voltages["t"]) / (1e-3 * noise) for name in names])
sigma = np.sqrt(np.diag(np.linalg.inv(sensitivity.T @ sensitivity)))
print(f"optimizer: {best.iterations} iterations, {best.run_count} circuit simulations, "
      f"χ² = N·MSD/s² = {times.size * best.objectives['msd'] / (1e3 * noise) ** 2:.1f} for {times.size - 3} degrees of freedom")
for name, label, unit, s in zip(names[:2], ("R0", "R1"), ("mΩ", "mΩ"), sigma):
    print(f"{label:4s} estimated {best.parameters[name]:8.4f} ± {s:.4f} {unit}, true {true[name]:6.2f} {unit}  "
          f"({(best.parameters[name] - true[name]) / s:+.2f} standard deviations)")
print(f"tau1 estimated {np.exp(best.parameters['ln_tau1']):8.4f} ± {np.exp(best.parameters['ln_tau1']) * sigma[2]:.4f} s, true 30.00 s  "
      f"({(best.parameters['ln_tau1'] - true['ln_tau1']) / sigma[2]:+.2f} standard deviations)")
print(f"rms voltage residual {1e3 * np.sqrt(np.mean((fitted.voltages['t'] - data) ** 2)):.3f} mV (noise {1e3 * noise:.1f} mV); "
      f"KCL residual of the circuit {np.abs(fitted.kcl_residual).max():.1e} A")

## Figure

The measured and fitted terminal voltage with the current profile; a close-up of one
pulse, where the instantaneous jump ($R_0$) and the relaxation ($R_1$, $\tau_1$)
separate; and the residuals (dotted: ± the noise level).

In [ ]:
fig = plots.figure(1, 3, size=(17, 5))
plots.line(times / 60, {"fitted model": fitted.voltages["t"]}, points={"pulse-test data": (times[::4] / 60, data[::4])},
           right={"current [A]": np.array([load(t) for t in times])}, xlabel="time [min]", ylabel="terminal voltage [V]",
           right_label="current [A] (discharge > 0)", title="Pulse test", legend="lower left", ax=fig[0])
window = (times >= 490) & (times <= 600)
plots.line(times[window], {"fitted model": fitted.voltages["t"][window]}, points={"pulse-test data": (times[window], data[window])},
           reference={"start values": pulse_test(start_values).solve(times=times, max_step=0.25).voltages["t"][window]},
           xlabel="time [s]", ylabel="terminal voltage [V]", title="Second pulse pair, close-up", legend="lower right", ax=fig[1])
plots.line(points={"data - model": (times / 60, 1e3 * (data - fitted.voltages["t"]))}, hlines=[1e3 * noise, -1e3 * noise],
           xlabel="time [min]", ylabel="voltage residual [mV]", title="Residuals", legend="upper right", ax=fig[2])
plots.show()

## Going further

Add a second RC branch with a longer relaxation time and refit: with these data its
parameters stay poorly determined, which the covariance shows at once. Make the
resistances functions of the state of charge, or fit the open-circuit voltage curve as
well from a slow (C/20) discharge. Real cycler data replace the synthetic `data`
array directly.